# Hyperparameter Tuning

Grid-searches XGBoost, LightGBM, RandomForest, and Ridge using time-series
cross-validation (expanding window). Scored on MAPE. Results are printed
as a ready-to-paste `PARAM_OVERRIDES` dict for `training.ipynb`.

**Does not touch Google Sheets or write anything** — read-only, safe to re-run.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import itertools
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge

import config
from forecasting.ingestion import load_series
from forecasting.features import build_features, FORECAST_FEATURE_COLS
from forecasting.metrics import mape

print('Setup complete.')

## 2. Configuration

Adjust `N_SPLITS` (number of CV folds) and `TEST_HOURS` (holdout per fold) to trade speed for thoroughness. `N_SPLITS=5` with `TEST_HOURS=720` (30 days) is a reasonable default.

In [ ]:
N_SPLITS   = 5      # number of expanding-window CV folds
TEST_HOURS = 24 * 30  # hours per fold holdout (~30 days)

# ── Grids ───────────────────────────────────────────────────────────────────
# Keep each grid small — combinatorial explosion is real.
# Add/remove values here; the search loop below handles the rest.

GRIDS = {
    'XGBoost': {
        'n_estimators':     [400, 600, 800],
        'learning_rate':    [0.02, 0.05, 0.10],
        'max_depth':        [4, 5, 6],
        'subsample':        [0.7, 0.8],
        'colsample_bytree': [0.7, 0.8],
        'min_child_weight': [3, 5, 10],
    },
    'LightGBM': {
        'n_estimators':      [400, 600, 800],
        'learning_rate':     [0.02, 0.05, 0.10],
        'num_leaves':        [31, 63, 127],
        'min_child_samples': [10, 20, 40],
        'subsample':         [0.7, 0.8],
        'colsample_bytree':  [0.7, 0.8],
    },
    'RandomForest': {
        'n_estimators':    [200, 400],
        'max_depth':       [8, 12, 16, None],
        'min_samples_leaf':[4, 8, 16],
        'max_features':    [0.5, 0.7, 'sqrt'],
    },
    'Ridge': {
        'alpha': [0.01, 0.1, 1.0, 10.0, 50.0, 100.0, 500.0],
    },
}

print('Grid sizes:')
for name, grid in GRIDS.items():
    n = 1
    for v in grid.values(): n *= len(v)
    print(f'  {name}: {n} combinations × {N_SPLITS} folds = {n * N_SPLITS} fits')

## 3. Load data & build features

In [ ]:
series = load_series()
print(f'Series: {series.shape[0]:,} hours  |  {series.index.min().date()} → {series.index.max().date()}')

feature_df = build_features(series)
X = feature_df[FORECAST_FEATURE_COLS]
y = feature_df['target']
print(f'Feature matrix: {X.shape[0]:,} rows × {X.shape[1]} cols')

## 4. Cross-validation search

Each model is grid-searched independently. Progress is printed per combination so you can see it working (and interrupt early if needed).

In [ ]:
tscv = TimeSeriesSplit(n_splits=N_SPLITS, test_size=TEST_HOURS)

def _cv_mape(estimator_fn, params: dict) -> float:
    """Mean MAPE across all CV folds for a given (estimator, params) pair."""
    scores = []
    for train_idx, test_idx in tscv.split(X):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
        model = estimator_fn(params)
        model.fit(X_tr, y_tr)
        y_pred = np.clip(model.predict(X_te), 0, None)
        scores.append(mape(y_te.values, y_pred))
    return float(np.mean(scores))

def _grid_search(name: str, estimator_fn, grid: dict) -> dict:
    keys   = list(grid.keys())
    values = list(grid.values())
    combos = list(itertools.product(*values))
    best_score  = float('inf')
    best_params = {}
    print(f'\n── {name} ({len(combos)} combos × {N_SPLITS} folds) ──')
    for i, combo in enumerate(combos, 1):
        params = dict(zip(keys, combo))
        score  = _cv_mape(estimator_fn, params)
        tag    = ' ← best' if score < best_score else ''
        print(f'  [{i:>4}/{len(combos)}]  MAPE={score:.3f}%  {params}{tag}')
        if score < best_score:
            best_score  = score
            best_params = params
    print(f'  Best: MAPE={best_score:.3f}%  →  {best_params}')
    return best_params, best_score

print('Helpers ready.')

In [ ]:
xgb_best, xgb_score = _grid_search(
    'XGBoost',
    lambda p: xgb.XGBRegressor(**p, random_state=42, verbosity=0),
    GRIDS['XGBoost'],
)

In [ ]:
lgb_best, lgb_score = _grid_search(
    'LightGBM',
    lambda p: lgb.LGBMRegressor(**p, random_state=42, verbosity=-1),
    GRIDS['LightGBM'],
)

In [ ]:
rf_best, rf_score = _grid_search(
    'RandomForest',
    lambda p: RandomForestRegressor(**p, random_state=42, n_jobs=1),
    GRIDS['RandomForest'],
)

In [ ]:
ridge_best, ridge_score = _grid_search(
    'Ridge',
    lambda p: Ridge(**p),
    GRIDS['Ridge'],
)

## 5. Results

Copy the `PARAM_OVERRIDES` block below into `training.ipynb` cell `b08a8017` (the configuration cell). The +Sales variants share the same base hyperparameters — they are duplicated in the output automatically.

In [ ]:
results = {
    'XGBoost':      (xgb_best,   xgb_score),
    'LightGBM':     (lgb_best,   lgb_score),
    'RandomForest': (rf_best,    rf_score),
    'Ridge':        (ridge_best, ridge_score),
}

# ── Summary table ────────────────────────────────────────────────────────────
print('=' * 60)
print(f'{"Model":<20}  {"CV MAPE":>10}  Best params')
print('=' * 60)
for name, (params, score) in results.items():
    print(f'{name:<20}  {score:>9.3f}%  {params}')

# ── Ready-to-paste PARAM_OVERRIDES ───────────────────────────────────────────
def _fmt_val(v):
    if v is None:          return 'None'
    if isinstance(v, str): return f'"{v}"'
    return repr(v)

def _fmt_dict(d, indent=8):
    pad = ' ' * indent
    inner = (',\n' + pad).join(f'"{k}": {_fmt_val(v)}' for k, v in d.items())
    return '{\n' + pad + inner + ',\n' + ' ' * (indent - 4) + '}'

static = {
    'HoltWinters':  '{"trend": "add", "seasonal": "add", "seasonal_periods": 168, "damped_trend": True}',
    'Prophet':      '{"weekly_seasonality": True, "daily_seasonality": True, "yearly_seasonality": False, "changepoint_prior_scale": 0.05}',
    'SeasonalNaive':'{"weeks": 8}',
}

print()
print('=' * 60)
print('PASTE INTO training.ipynb → PARAM_OVERRIDES cell:')
print('=' * 60)
print()
print('PARAM_OVERRIDES = {')
for name, (params, _) in results.items():
    print(f'    "{name}": {_fmt_dict(params)},')
for name, val in static.items():
    print(f'    "{name}": {val},')
print()
print('    # +Sales variants — same base hyperparameters')
for name, (params, _) in results.items():
    sales_name = name + '+Sales'
    print(f'    "{sales_name}": {_fmt_dict(params)},')
print('    "Prophet+Sales": {"weekly_seasonality": True, "daily_seasonality": True, "yearly_seasonality": False, "changepoint_prior_scale": 0.05},')
print('}')